# 05 Matrix imaging

For every array design from 02: simulate what it would record over a known BDT, image it, and compare with the
truth. The correlations are **fully converged**: every pair of stations that records at the same time gives the
exact echo, with no noise. This is the best each design can do; 06 adds noise.

- **Earth:** the velocity model of 01 with a dome-shaped BDT zone (Vp 5 % higher inside) under the BDT high in
  `SCENARIOS`. The dome replaces the scenarios of the old 04.
- **Echoes:** the BDT zone as Born scatterers; the reflection matrix R(f), 1-6 Hz, between every pair of stations
  that record together (Trilliums, Mercury and GeoNet all deployment; nodes during their phase).
- **Image:** confocal image, focused with the 3D travel times of 03.
- **Phased designs** run as in the field: phase 1 is imaged on its own, the BDT high is picked from it, and phase 2
  is laid out around the pick with 02's rules.
- **Scores:** the BDT top is picked in each image column as the shallowest strong peak below 3 km (`pick_z`;
  the image itself starts at 0.5 km). `area_coverage` is the fraction of the MT target area (4 km around the drill
  site) where that pick is within 0.3 km of the truth; `crest_imaged_km` is the depth of the brightest point above the
  true crest; `target_error_km` is how far the picked BDT high is from the true one.

The forward model, imaging and scoring are in `mi_common.py`, shared with 06.

Inputs: `data/velocity_model.npz`, `data/stations.csv`, `data/designs.json`, `data/tt/`.
Outputs: `data/results_05_mi.csv`, images in `data/mi_images_05/`

In [ ]:
# --- Imports ---
import os
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from mi_common import MatrixImaging

In [ ]:
# --- Settings ---
SCENARIOS = {"dome": (0.0, 0.0)}      # BDT high (km E, N of the drill site). Add more, e.g. "dome 3 km E": (3.0, 0.0)
DESIGNS = None                         # None = every design in designs.json
OUT = "../data/mi_images_05"
RESULTS = "../data/results_05_mi.csv"

mi = MatrixImaging(scenarios=SCENARIOS)
DESIGNS = DESIGNS or mi.designs()
os.makedirs(OUT, exist_ok=True)
print(mi)

## The Earth

Vp along the node axis (AZ) through the BDT high, with the BDT zone outlined, and the depth of its top in map view.

In [ ]:
# --- Plot styles (as the 02 map) ---
ST = mi.stations
STY = dict(
    rt=dict(marker="^", mfc="#ea6a2c", mec="#3d2414", mew=0.8, ms=7, ls="none"),
    ns=dict(marker="v", mfc="#1f9e78", mec="#14332a", mew=0.8, ms=7, ls="none"),
    gn=dict(marker="D", mfc="#6a3d9a", mec="#2b1640", mew=0.8, ms=5.5, ls="none"),
    tr=dict(marker="s", mfc="none", mec="#2a78d6", mew=1.6, ms=7, ls="none"),
    drill=dict(marker="*", mfc="#f5c518", mec="#0b0b0b", mew=0.8, ms=13, ls="none"),
    n1=dict(marker="o", mfc="white", mec="#0b0b0b", mew=1.4, ms=5.5, ls="none"),
    n2=dict(marker="o", mfc="white", mec="#0b0b0b", mew=0.7, ms=3.5, ls="none"),
)
GROUPS = {"rt": (ST.type == "mercury") & ST.name.fillna("").str.startswith("RT"),
          "ns": (ST.type == "mercury") & ST.name.fillna("").str.startswith("NS"),
          "gn": ST.type == "geonet", "tr": ST.type == "trillium"}
PATCH_COL = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#6250d6"]
th = np.linspace(0, 2 * np.pi, 200)
plt.rcParams["hatch.linewidth"] = 0.8


def stations_map(ax, small=False):
    ax.plot(mi.MT_R * np.cos(th), mi.MT_R * np.sin(th), "--", color="#3d3c39", lw=1.1)
    for k, sel in GROUPS.items():
        ax.plot(ST.x[sel], ST.y[sel], **({**STY[k], "ms": STY[k]["ms"] * 0.7} if small else STY[k]))
    ax.plot(0, 0, **STY["drill"])


def bdt_zone(ax, scenario, alpha=0.7):
    s, _, _, top, base = mi.velocity_section(scenario)
    ax.fill_between(s, top, base, facecolor="none", edgecolor=(1, 1, 1, alpha), hatch="///", lw=0)
    ax.plot(s, top, "--", color="white", lw=1.5)
    ax.plot(s, base, ":", color="white", lw=1.5)


R_MAP = mi.IX[-1]
for name in SCENARIOS:
    s, z, vp, top, base = mi.velocity_section(name)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 4.6), width_ratios=[1.6, 1])
    pc = a1.pcolormesh(s, z, vp.T, cmap="viridis", shading="auto", rasterized=True)
    bdt_zone(a1, name)
    a1.set(ylim=(9, 0), xlabel="Distance along the node axis (km)", ylabel="Depth (km)",
           title=f"{name}: Vp with the BDT zone (Vp {mi.DVP:+.0%})", aspect=1)
    fig.colorbar(pc, ax=a1, label="Vp (km/s)", shrink=0.8)
    pt = a2.pcolormesh(mi.IX, mi.IY, mi.true_top(name).T, cmap="viridis_r", shading="auto")
    stations_map(a2)
    _, lx, ly = mi.line(name)
    a2.plot(lx, ly, "-", color="#e34948", lw=1.5)
    a2.set(aspect="equal", xlim=(-R_MAP, R_MAP), ylim=(-R_MAP, R_MAP), xlabel="East (km)", ylabel="North (km)",
           title="BDT top; red: section line")
    fig.colorbar(pt, ax=a2, label="BDT top (km)", shrink=0.8).ax.invert_yaxis()
    plt.tight_layout()
    plt.show()

## Run every design

Saved images are reused on a rerun (delete `data/mi_images_05/` after changing 02, 03 or the settings here).

In [ ]:
# --- Image every design for every scenario ---
slug = lambda s: re.sub(r"[^a-z0-9]+", "_", s.lower()).strip("_")
rows = []
for name in SCENARIOS:
    for design in DESIGNS:
        f = f"{OUT}/{slug(name)}__{slug(design)}.npz"
        r = np.load(f, allow_pickle=True) if os.path.exists(f) else None
        if r is None or r["image"].shape != (len(mi.IX), len(mi.IY), len(mi.IZ)):
            out = mi.run_design(design, name)
            nodes = [(s, q) for q, p in enumerate(out["phases"]) for s in p["stations"]]
            info = dict(stations=len(out["ids"]), pairs=out["n_pairs"], seconds=round(out["seconds"]))
            np.savez_compressed(f, image=out["image"], phase1=out.get("phase1", np.zeros(0)),
                                target=out.get("target", np.full(2, np.nan)), node_ids=[s for s, _ in nodes],
                                node_dep=[q for _, q in nodes], info=np.array(info, dtype=object))
            r = np.load(f, allow_pickle=True)
            print(f"{name} / {design}: {out['seconds']:.0f} s", flush=True)
        rows.append(dict(scenario=name, design=design, **r["info"].item(), **mi.score(r["image"], name)))  # scored fresh
res = pd.DataFrame(rows)
res.to_csv(RESULTS, index=False)
res.set_index(["scenario", "design"]).drop(columns="seconds").round(2)

## Recovered images

Sections along the node axis through the BDT high, at true scale. The hatched band is the true BDT zone (top
dashed, base dotted); each image is normalised to its own maximum.

In [ ]:
# --- Sections: one panel per design ---
load = lambda name, design: np.load(f"{OUT}/{slug(name)}__{slug(design)}.npz", allow_pickle=True)

for name in SCENARIOS:
    n = len(DESIGNS)
    nc = 2
    fig, axs = plt.subplots(int(np.ceil(n / nc)), nc, figsize=(12, 3.0 * np.ceil(n / nc)), sharex=True, sharey=True,
                            squeeze=False, layout="constrained")
    for ax, design in zip(axs.ravel(), DESIGNS):
        r = load(name, design)
        s, z, sec = mi.section(r["image"], name)
        pm = ax.pcolormesh(s, z, sec.T, cmap="magma", vmin=0, vmax=1, shading="auto", rasterized=True)
        bdt_zone(ax, name, alpha=0.45)
        sc = res.set_index(["scenario", "design"]).loc[(name, design)]
        ax.set(ylim=(mi.IZ[-1], 0), aspect=1)
        ax.set_title(f"{design}   (MT-area coverage {sc['area_coverage']:.2f}, crest at {sc['crest_imaged_km']:.1f} km)",
                     loc="left", fontsize=10)
    for ax in axs.ravel()[n:]:
        ax.axis("off")
    for ax in axs[:, 0]:
        ax.set_ylabel("Depth (km)")
    for ax in axs[-1]:
        ax.set_xlabel("Distance along the node axis (km)")
    fig.colorbar(pm, ax=axs, label="Image amplitude (normalised)", shrink=0.5)
    fig.suptitle(f"{name}: fully converged correlations", fontsize=12)
    plt.show()

In [ ]:
# --- Map view: picked BDT top minus true (white = right, red = too shallow, blue = too deep, grey = not imaged) ---
def nodes_map(ax, r, design):
    st, dep = r["node_ids"], r["node_dep"]
    if mi.is_phased(design):
        ax.plot(ST.x[st[dep == 0]], ST.y[st[dep == 0]], **STY["n1"])
        for q in range(1, dep.max() + 1):
            ax.plot(ST.x[st[dep == q]], ST.y[st[dep == q]], "o", ms=3.2, mfc=PATCH_COL[(q - 1) % 6], mec="#0b0b0b",
                    mew=0.4)
    else:
        ax.plot(ST.x[st], ST.y[st], **STY["n2"])


for name in SCENARIOS:
    n = len(DESIGNS)
    nc = 3
    fig, axs = plt.subplots(int(np.ceil(n / nc)), nc, figsize=(15, 5 * np.ceil(n / nc)), sharex=True, sharey=True,
                            squeeze=False, layout="constrained")
    for ax, design in zip(axs.ravel(), DESIGNS):
        r = load(name, design)
        err = mi.pick_top(r["image"]) - mi.true_top(name)
        ax.set_facecolor("#d9d8d3")
        pc = ax.pcolormesh(mi.IX, mi.IY, err.T, cmap="RdBu", vmin=-1, vmax=1, shading="auto", rasterized=True)
        stations_map(ax, small=True)
        nodes_map(ax, r, design)
        ax.set(aspect="equal", xlim=(-R_MAP, R_MAP), ylim=(-R_MAP, R_MAP))
        ax.set_title(f"{design}   ({res.set_index(['scenario', 'design']).loc[(name, design), 'area_coverage']:.2f})",
                     loc="left", fontsize=10)
    for ax in axs.ravel()[n:]:
        ax.axis("off")
    fig.colorbar(pc, ax=axs, label="Picked minus true BDT top (km)", shrink=0.5)
    fig.suptitle(f"{name}: where each design recovers the BDT top (dashed: MT target area)", fontsize=12)
    plt.show()

## Phased designs: from phase 1 to phase 2

Left: the phase 1 image alone, with the BDT high picked from it (red +) and the true one (gold star). Right: the
final image, with phase 2 laid out around the pick.

In [ ]:
# --- Phase 1 pick and where phase 2 went ---
phased = [d for d in DESIGNS if mi.is_phased(d)]
for name in SCENARIOS:
    c = mi.centre(name)
    for design in phased:
        r = load(name, design)
        fig, axs = plt.subplots(1, 3, figsize=(16, 4.6), width_ratios=[1, 1.3, 1.3], layout="constrained")
        top1 = mi.pick_top(r["phase1"])
        axs[0].set_facecolor("#d9d8d3")
        axs[0].pcolormesh(mi.IX, mi.IY, (top1 - mi.true_top(name)).T, cmap="RdBu", vmin=-1, vmax=1, shading="auto")
        stations_map(axs[0], small=True)
        nodes_map(axs[0], r, design)
        axs[0].plot(*r["target"], "+", color="#e34948", ms=16, mew=3)
        axs[0].plot(*c, "*", mfc="#f5c518", mec="k", ms=14)
        axs[0].set(aspect="equal", xlim=(-R_MAP, R_MAP), ylim=(-R_MAP, R_MAP),
                   title=f"phase 1 image; + pick ({np.hypot(*(r['target'] - c)):.2f} km off), phase 2 around it")
        for ax, img, t in ((axs[1], r["phase1"], "phase 1 alone"), (axs[2], r["image"], "phase 1 + phase 2")):
            s, z, sec = mi.section(img, name)
            ax.pcolormesh(s, z, sec.T, cmap="magma", vmin=0, vmax=1, shading="auto", rasterized=True)
            bdt_zone(ax, name, alpha=0.45)
            ax.set(ylim=(mi.IZ[-1], 0), aspect=1, title=t, xlabel="Distance along the node axis (km)")
        axs[1].set_ylabel("Depth (km)")
        fig.suptitle(f"{name}: {design}", fontsize=12)
        plt.show()